#MVP - Sprint: Engenharia de Dados

**Nome Completo:** Raphael Morgado Rosenburg Henriques  
**Matrícula:** 4052026001014  

Link do dataset público: https://datariov2-pcrj.hub.arcgis.com/datasets/PCRJ::itbi-transa%C3%A7%C3%B5es-por-logradouro-e-m%C3%AAs-im%C3%B3veis-residenciais-e-n%C3%A3o-residenciais/about

In [0]:
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.functions import col, count, when
from pyspark.sql.functions import trim, upper, regexp_replace
from pyspark.sql.functions import monotonically_increasing_id

### 1. Camada Bronze: Ingestão e Persistência dos Dados Brutos


In [0]:
caminho_arquivo_csv = "/Volumes/workspace/default/raw-data/ITBI_-_Transa%C3%A7%C3%B5es_por_Logradouro_e_M%C3%AAs_-_Im%C3%B3veis_Residenciais_e_N%C3%A3o_Residenciais.csv"

#Leitura do CSV com os dados brutos
df_dados_brutos = spark.read.csv(caminho_arquivo_csv, header=True, inferSchema=False, sep=",")

#Metadados de auditoria
df_bronze = df_dados_brutos.withColumn("ingestao", current_timestamp()).withColumn("arquivo", lit("ITBI_Transacoes_Logradouro_Mes.csv"))

#Persistência na camada Bronze em formato Delta
df_bronze.write.mode("overwrite").format("delta").saveAsTable("bronze_imoveis")

#Visualização das primeiras 5 linhas da tabela
display(spark.table("bronze_imoveis").limit(10))


Nesta primeira etapa foi realizada a **ingestão dos dados, em seus formatos brutos, em um DataFrame PySpark**, estes que foram extraídos do seu arquivo CSV original armazenado no Volume criado no Unity Catalog.

Esta etapa é referente à **Camada Bronze** da Arquitetura Medalhão, por isso o nome do nosso DataFrame **df_bronze**. 

**Decisões de Engenharia utilizadas:**
* **Preservação de Integridade:** Na leitura do CSV foi utilizado o parâmetro *'inferSchema=False'* afim de garantir que todos os atributos fossem ingeridos com seus formatos originais.
* **Governança e Auditoria:** Foram criadas duas novas colunas de metadados assegurando maior controle à rastreabilidade temporal e a proveniência dos registros, sendo elas respectivamente, "ingestao" (current_timestamp()) e "arquivo" (lit("ITBI_Transacoes_Logradouro_Mes.csv")).
* **Delta Lake:** Os dados foram persistidos na tabela "bronze_imoveis" como o modo de escrita *'.write.mode("overwrite")'* e com formato *'.format("delta")'*, habilitando propriedades transacionais ACID e versionamento nativo do Lakehouse.

In [0]:
spark.table("bronze_imoveis").printSchema()

### 2. Diagnóstico e Qualidade dos Dados

In [0]:
#Total de registros
total_linhas = df_bronze.count()
print(f"Total de registros na camada Bronze: {total_linhas}")

#Verificação de Unicidade/Duplicatas
unicos_id = df_bronze.select("objectid").distinct().count()
print(f"Total de IDs únicos (objectid): {unicos_id}")
print(f"Total de duplicatas: {total_linhas - unicos_id}")

#Verificação de Valores Nulos por atributo
valores_nulos = df_bronze.select([count(when(col(c).isNull(), c)).alias(c) for c in df_bronze.columns]).first().asDict()
    
print("\nValores nulos/ausentes detectados:")
for coluna, qtd in valores_nulos.items():
    if qtd > 0:
        print(f"  - {coluna}: {qtd}")



Na célula acima foram realizadas verificações fundamentais para mensurar a integridade da base bruta (`bronze_imoveis`):

* **Unicidade:** O atributo identificador *'objectid'* possui 97.467 registros únicos para um total de 97.467 linhas, confirmando **zero duplicatas** de chave primária.
* **Completude:** Identificou-se que praticamente todas as colunas estão íntegras, com exceção de **4 registros nulos/vazios** concentrados no atributo *'média_valor_imóvel'* (menos de 0,004% da base).
* **Consistência e Tipagem:** Constatou-se a necessidade de conversão dos tipos primitivos (de texto para inteiros e decimais com ponto), padronização textual em maiúsculas e saneamento dos nomes das colunas, etapas que serão tratadas na construção da **Camada Silver**.

### 3. Camada Silver: Limpeza, Tipagem e Tratamento de Qualidade

In [0]:
df_bronze = spark.table("workspace.default.bronze_imoveis")

#Saneamento de nomes, conversão de tipos, tratamento de decimais e inconsistências e padronização textual
df_silver = df_bronze \
    .withColumn("id_transacao", col("objectid").cast("long")) \
    .withColumn("codigo_logradouro", col("cl").cast("integer")) \
    .withColumn("logradouro", trim(upper(col("logradouro")))) \
    .withColumn("codigo_bairro", trim(col("codbairro"))) \
    .withColumn("bairro", trim(upper(col("bairro")))) \
    .withColumn("tipo_uso", trim(upper(col("uso")))) \
    .withColumn("tipologia_principal", trim(upper(col("principais_tipologias")))) \
    .withColumn("transacao_mercado", trim(upper(col("principal_transação_mercado")))) \
    .withColumn("ano", col("ano_transação").cast("integer")) \
    .withColumn("mes", col("mês_transação").cast("integer")) \
    .withColumn("total_transacoes", col("total_transações").cast("integer")) \
    .withColumn("percentual_transferido_medio", regexp_replace(col("média_percentual_transferido"), ",", ".").cast("double")) \
    .withColumn("area_construida_media", regexp_replace(col("média_área_construída"), ",", ".").cast("double")) \
    .withColumn("valor_transacao_medio", regexp_replace(col("média_valor_transação"), ",", ".").cast("double")) \
    .withColumn("valor_imovel_medio", regexp_replace(col("média_valor_imóvel"), ",", ".").cast("double")) \
    .filter(col("id_transacao").isNotNull() & (col("ano") > 1900) & col("valor_imovel_medio").isNotNull())
    .select("id_transacao", "codigo_logradouro", "logradouro", "codigo_bairro", "bairro", "tipo_uso", "tipologia_principal", "transacao_mercado", "ano", "mes", "total_transacoes", "percentual_transferido_medio", "area_construida_media", "valor_transacao_medio", "valor_imovel_medio", "ingestao")

#Persistência na Camada Silver no formato Delta Lake
df_silver.write.mode("overwrite").format("delta").saveAsTable("silver_imoveis")

#Exibição de 10 linhas aleatórias para validação
display(spark.table("silver_imoveis").sample(fraction=0.1).limit(10))



Nesta etapa, os dados brutos da tabela **'bronze_imoveis'** foram padronizados para garantir consistência semântica e analítica:

* **Saneamento de Nomenclaturas:** Atributos renomeados para o padrão *snake_case*, eliminando acentos e caracteres incompatíveis.
* **Tipagem Estrita (cast):** Conversão de strings brutas para **long**, **integer** e **double**.
* **Tratamento de Decimais:** Substituição de vírgula por ponto, pelo parâmetro (*'regexp_replace'*) em métricas monetárias e de área, visto que o Spark só reconhece números decimais com pontos.
* **Padronização Textual:** Aplicação dos parâmetros *'trim()'* e *'upper()'* nos campos categóricos para evitar discrepâncias em agrupamentos.
* **Tratamento de Inconsistências:** Expurgo de anos inválidos e remoção dos **4 registros nulos** encontrados na análise anterior no campo *'valor_imovel_medio'*.
* **Persistência:** Tabela salva em formato Delta Lake como  **"silver_imoveis"**.

### 4. Camada Gold: Modelagem Dimensional em Esquema Estrela


In [0]:
df_silver = spark.table("silver_imoveis")

#Criação das Dimensões
##Dimensão Localização
dim_localizacao = df_silver.select("codigo_logradouro", "logradouro", "codigo_bairro", "bairro").distinct().withColumn("id_localizacao", monotonically_increasing_id())

dim_localizacao.write.mode("overwrite").format("delta").saveAsTable("dim_localizacao")

##Dimensão Tipologia
dim_tipologia = df_silver.select("tipo_uso", "tipologia_principal", "transacao_mercado").distinct().withColumn("id_tipologia", monotonically_increasing_id())

dim_tipologia.write.mode("overwrite").format("delta").saveAsTable("dim_tipologia")

##Dimensão Tempo
dim_tempo = df_silver.select("ano", "mes").distinct().withColumn("id_tempo", monotonically_increasing_id())

dim_tempo.write.mode("overwrite").format("delta").saveAsTable("dim_tempo")

##Tabela Fato Transações
fato_transacoes = df_silver.join(dim_localizacao, on=["codigo_logradouro", "logradouro", "codigo_bairro", "bairro"], how="inner") \
    .join(dim_tipologia, on=["tipo_uso", "tipologia_principal", "transacao_mercado"], how="inner") \
    .join(dim_tempo, on=["ano", "mes"], how="inner") \
    .select("id_transacao", "id_localizacao", "id_tipologia", "id_tempo", "total_transacoes", "percentual_transferido_medio", "area_construida_media", "valor_transacao_medio", "valor_imovel_medio")

fato_transacoes.write.mode("overwrite").format("delta").saveAsTable("fato_transacoes")

#Validação das Tabelas
display(spark.table("fato_transacoes").limit(10))

display(spark.table("dim_localizacao").limit(5))
display(spark.table("dim_tipologia").limit(5))
display(spark.table("dim_tempo").limit(5))


Nesta etapa, os dados tratados da camada Silver foram estruturados seguindo o padrão clássico de **Esquema Estrela (*Star Schema*)**, otimizando o repositório para consultas analíticas e separando entidades de contexto das métricas quantitativas:

* **dim_localizacao:** Dimensão contendo os atributos geográficos (*'codigo_logradouro'*, *'logradouro'*, *'codigo_bairro'*, *'bairro'*), indexada pela chave primária **'id_localizacao'**.
* **dim_tipologia:** Dimensão descritiva dos segmentos imobiliários (*'tipo_uso'*, *'tipologia_principal'*, *'transacao_mercado'*), indexada por **'id_tipologia'**.
* **dim_tempo:** Dimensão temporal contendo as partições de *'ano'* e *'mes'*, indexada por **'id_tempo'**.
* **fato_transacoes:** Tabela fato central relacionando as métricas de negócio (*'total_transacoes'*, *'percentual_transferido_medio'*, *'area_construida_media'*, *'valor_transacao_medio'*, *'valor_imovel_medio'*) às suas respectivas dimensões através de **chaves estrangeiras** (**'id_localizacao'**, **'id_tipologia'**, **'id_tempo'**).

### 5. Catálogo de Dados

In [0]:
%sql
--Dimensão Localização-------------------------------------------------------------------------------------------------------------------------------------------------------------- 
COMMENT ON TABLE dim_localizacao IS 'Dimensão de Localização Geográfica. Contém logradouros, bairros e seus códigos cadastrais do município do Rio de Janeiro. Originada da camada SILVER (silver_imoveis)';

COMMENT ON COLUMN dim_localizacao.id_localizacao IS 'Chave Primária (PK) gerada monotonicamente (monotonically_increasing_id) para indexação dimensional. Tipo de Dado: Long. Domínio: Inteiros >= 0.';
COMMENT ON COLUMN dim_localizacao.codigo_logradouro IS 'Código do logradouro cadastrado no município do Rio de Janeiro (CL). Tipo de Dado: Integer (Número Inteiro). Domínio: Inteiros Positivos (>0).';
COMMENT ON COLUMN dim_localizacao.logradouro IS 'Nome oficial do logradouro em caixa alta. Tipo de Dado: String (Texto). Domínio: Nomes válidos de vias públicas.';
COMMENT ON COLUMN dim_localizacao.codigo_bairro IS 'Código do bairro cadastrado no município do Rio de Janeiro (CB). Tipo de Dado: String (Texto). Domínio: Códigos numéricos de 3 dígitos (ex: 001 a 160).';
COMMENT ON COLUMN dim_localizacao.bairro IS 'Nome oficial do bairro do Rio de Janeiro em caixa alta. Tipo de Dado: String (Texto). Domínio: Bairros oficiais cadastrados do município do Rio de Janeiro.';   

--Dimensão Tipologia-----------------------------------------------------------------------------------------------------------------------------------------------------------------
COMMENT ON TABLE dim_tipologia IS 'Dimensão de Classificação Imobiliária. Contém os tipos de uso, tipologias e transações cadastrados no município do Rio de Janeiro. Originada da camada SILVER (silver_imoveis)';

COMMENT ON COLUMN dim_tipologia.id_tipologia IS 'Chave Primária (PK) gerada monotonicamente (monotonically_increasing_id) para indexação dimensional. Tipo de Dado: Long. Domínio: Inteiros >= 0.';
COMMENT ON COLUMN dim_tipologia.tipo_uso IS 'Finalidade de utilização do imóvel. Tipo de Dado: String (Texto). Domínio: [RESIDENCIAL, NÃO RESIDENCIAL].';
COMMENT ON COLUMN dim_tipologia.tipologia_principal IS 'Classificação construtica do imóvel. Tipo de Dado: String (Texto). Domínio: [APARTEMENTO, CASA, SALA, LOJA, PRÉDIO, GALPÃO, etc].';
COMMENT ON COLUMN dim_tipologia.transacao_mercado IS 'Natureza jurídica da transmissão imobiliária. Tipo de Dado: String (Texto). Domínio: [COMPRA E VENDA, ALUGUEL, DOAÇÃO, etc].';

--Dimensão Tempo-------------------------------------------------------------------------------------------------------------------------------------------------------------------- 
COMMENT ON TABLE dim_tempo IS 'Dimensão de Tempo. Contém os anos e mêses de incidência da transação dos imóveis. Originada da camada SILVER (silver_imoveis)';

COMMENT ON COLUMN dim_tempo.id_tempo IS 'Chave Primária (PK) gerada monotonicamente (monotonically_increasing_id) para indexação dimensional. Tipo de Dado: Long. Domínio: Inteiros >= 0.';
COMMENT ON COLUMN dim_tempo.ano IS 'Ano de registro da transação. Tipo de Dado: Integer (Número Inteiro). Domínio: Anos >= 2000.';
COMMENT ON COLUMN dim_tempo.mes IS 'Mês de registro da transação. Tipo de Dado: Integer (Número Inteiro). Domínio: Mês entre 1 e 12.';

--Tabela Fato------------------------------------------------------------------------------------------------------------------------------------------------------------------------
COMMENT ON TABLE fato_transacoes IS 'Tabela Fato de Transações Imobiliárias (ITBI). Armazena métricas transacionais, áreas e valores médios por logradouro e período. Originada pela junção (JOIN) da "silver_imoveis" com as dimensões "dim_localizacao", "dim_tipologia" e "dim_tempo".';

COMMENT ON COLUMN fato_transacoes.id_transacao IS 'Identificador analítico único da transação, Chave Primária (PK). Tipo de Dado: Long. Domínio: Inteiros >= 0.';
COMMENT ON COLUMN fato_transacoes.id_localizacao IS 'Chave Estrangeira (FK) referenciando dim_localizacao.id_localizacao. Tipo de Dado: Long. Domínio: IDs presentes na dimensão localização.';
COMMENT ON COLUMN fato_transacoes.id_tipologia IS 'Chave Estrangeira (FK) referenciando dim_tipologia.id_tipologia. Tipo de Dado: Long. Domínio: IDs presentes na dimensão tipologia.';
COMMENT ON COLUMN fato_transacoes.id_tempo IS 'Chave Estrangeira (FK) referenciando dim_tempo.id_tempo. Tipo de Dado: Long. Domínio: IDs presentes na dimensão tempo.';
COMMENT ON COLUMN fato_transacoes.total_transacoes IS 'Contagem consolidada de transações para o logradouro no período. Tipo de Dado: Integer (Número Inteiro). Domínio: Inteiros >= 1.';
COMMENT ON COLUMN fato_transacoes.percentual_transferido_medio IS 'Percentual médio de propriedade transferido. Tipo de Dado: Decimal (Double). Domínio: Valores contínuos de 0.0 a 100.0.';
COMMENT ON COLUMN fato_transacoes.area_construida_media IS 'Área construída média das unidades em metros quadrados (m²). Tipo de Dado: Decimal (Double). Domínio: Valores reais > 0.0.';
COMMENT ON COLUMN fato_transacoes.valor_transacao_medio IS 'Valor médio efetivo declarado da transação em Reais (BRL). Tipo de Dado: Decimal (Double). Domínio: Valores monetários > 0.0.';
COMMENT ON COLUMN fato_transacoes.valor_imovel_medio IS 'Valor venal/avaliado médio do imóvel apurado pela prefeitura em Reais (BRL). Tipo de Dado: Decimal (Double). Domínio: Valores monetários > 0.0.';
    

### 6. Análise de Dados 

Para validar a utilidade analítica do repositório de dados estruturado no modelo **Esquema Estrela (Camada Gold)**, serão respondidas 5 perguntas:

1. **Quais são os 10 bairros com maior volume transacional registrados no município do Rio de Janeiro?**
2. **Quais são os 10 bairros com o valor médio de transação imobiliária mais elevado?**
3. **Como o volume e o montante financeiro das transações imobiliárias evoluíram historicamente ano a ano no Rio de Janeiro?**
4. **Qual tipologia construtiva (Apartamento, Casa, Loja) e finalidade de uso (Residencial vs. Comercial) dominam o mercado imobiliário carioca?**
5. **Qual é o valor médio transacionado de apartamentos residenciais nos bairros que concentram o maior volume de vendas dessa categoria?**

####1. **Quais são os 10 bairros com maior volume transacional registrados no município do Rio de Janeiro?**

In [0]:
%sql
SELECT l.bairro AS Bairro, SUM(f.total_transacoes) AS Total_de_Transacoes_Registradas FROM fato_transacoes f
JOIN dim_localizacao l ON f.id_localizacao = l.id_localizacao
GROUP BY l.bairro
ORDER BY Total_de_Transacoes_Registradas DESC
LIMIT 10;

Databricks visualization. Run in Databricks to view.

####2. **Quais são os 10 bairros com o valor médio de transação imobiliária mais elevado?**

In [0]:
%sql
SELECT l.bairro AS Bairro, ROUND(AVG(f.valor_transacao_medio), 2) AS Valor_Medio_Transacao, ROUND(AVG(f.valor_imovel_medio), 2) AS Valor_Venal_Medio, SUM(f.total_transacoes) AS Volume_Amostral FROM fato_transacoes f
JOIN dim_localizacao l ON f.id_localizacao = l.id_localizacao
GROUP BY l.bairro
HAVING Volume_Amostral >= 100
ORDER BY Valor_Medio_Transacao DESC
LIMIT 10;

Databricks visualization. Run in Databricks to view.

####3. **Como o volume e o montante financeiro das transações imobiliárias evoluíram historicamente ano a ano no Rio de Janeiro?**

In [0]:
%sql
SELECT t.ano, SUM(f.total_transacoes) AS Volume_Transacional_Anual, ROUND(AVG(f.valor_transacao_medio), 2) AS Ticket_Medio_Anual FROM fato_transacoes f
JOIN dim_tempo t ON f.id_tempo = t.id_tempo
GROUP BY t.ano
ORDER BY t.ano ASC;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

####4. **Qual tipologia construtiva (Apartamento, Casa, Loja) e finalidade de uso (Residencial vs. Comercial) dominam o mercado imobiliário carioca?**

In [0]:
%sql
SELECT p.tipo_uso, p.tipologia_principal, SUM(f.total_transacoes) AS Total_Transacoes, ROUND(AVG(f.area_construida_media), 2) AS Area_m2, ROUND(AVG(f.valor_transacao_medio), 2) AS Valor_Medio_Transacao FROM fato_transacoes f
JOIN dim_tipologia p ON f.id_tipologia = p.id_tipologia
GROUP BY p.tipo_uso, p.tipologia_principal
ORDER BY Total_Transacoes DESC
LIMIT 10;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

####5. **Qual é o valor médio transacionado de apartamentos residenciais nos bairros que concentram o maior volume de vendas dessa categoria?**

In [0]:
%sql
SELECT l.bairro, ROUND(AVG(f.valor_transacao_medio), 2) AS Valor_Medio_Apartamento, ROUND(AVG(f.area_construida_media), 2) AS Area_m2, SUM(f.total_transacoes) AS Volume_Apartamentos FROM fato_transacoes f
JOIN dim_localizacao l ON f.id_localizacao = l.id_localizacao
JOIN dim_tipologia p ON f.id_tipologia = p.id_tipologia
WHERE p.tipo_uso = 'RESIDENCIAL' AND p.tipologia_principal = 'APARTAMENTO'
GROUP BY l.bairro
ORDER BY Volume_Apartamentos DESC
LIMIT 10;

Databricks visualization. Run in Databricks to view.